In [ ]:
import copy
from heapq import heappush, heappop

n = 3

# Hướng di chuyển: Dưới, Trái, Trên, Phải
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class priorityQueue:
    def __init__(self):
        self.heap = []

    def push(self, key): # Sửa seft -> self
        heappush(self.heap, key)

    def pop(self):
        return heappop(self.heap)

    def empty(self):
        return not self.heap

class nodes:
    def __init__(self, parent, mats, empty_tile_posi, costs, levels):
        self.parent = parent
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.costs = costs
        self.levels = levels

    # So sánh để priority queue biết node nào "nhỏ" hơn (ưu tiên hơn)
    def __lt__(self, nxt):
        return (self.costs + self.levels) < (nxt.costs + nxt.levels)

# Hàm tính số ô sai vị trí (Heuristic)
def calculateCosts(mats, final) -> int:
    count = 0
    for i in range(n):
        for j in range(n):
            # Nếu ô hiện tại khác 0 và khác vị trí đích
            if mats[i][j] != 0 and mats[i][j] != final[i][j]:
                count += 1
    return count # Phải nằm ngoài vòng lặp for

def newNode(mats, empty_tile_posi, new_empty_tile_posi, level, parent, final) -> nodes:
    new_mats = copy.deepcopy(mats)

    x1, y1 = empty_tile_posi
    x2, y2 = new_empty_tile_posi

    # Hoán đổi ô trống
    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    costs = calculateCosts(new_mats, final)
    return nodes(parent, new_mats, new_empty_tile_posi, costs, level)

def printMatrix(mats): # Đổi tên cho chuẩn
    for i in range(n):
        for j in range(n):
            print("%d " % (mats[i][j]), end=" ")
        print()

def isSafe(x, y):
    return 0 <= x < n and 0 <= y < n

def printPath(root):
    if root is None:
        return
    printPath(root.parent) # Đệ quy trước để in từ gốc
    printMatrix(root.mats)
    print("Cost: %d, Level: %d" % (root.costs, root.levels))
    print()

def solve(initial, empty_tile_posi, final):
    pq = priorityQueue()

    costs = calculateCosts(initial, final)
    root = nodes(None, initial, empty_tile_posi, costs, 0)

    pq.push(root)

    while not pq.empty():
        minimum = pq.pop()

        # Nếu số ô sai bằng 0 tức là đã xong
        if minimum.costs == 0:
            print("--- Tìm thấy lời giải ---")
            printPath(minimum)
            return

        # Sinh các node con
        for i in range(4):
            new_tile_posi = [
                minimum.empty_tile_posi[0] + rows[i],
                minimum.empty_tile_posi[1] + cols[i],
            ]

            if isSafe(new_tile_posi[0], new_tile_posi[1]):
                child = newNode(minimum.mats,
                                minimum.empty_tile_posi,
                                new_tile_posi,
                                minimum.levels + 1,
                                minimum, final)
                pq.push(child)

# --- PHẦN CHẠY CHƯƠNG TRÌNH ---
# Để bên ngoài tất cả các hàm và lớp
if __name__ == "__main__":
    initial = [[1, 2, 3],
               [5, 6, 0],
               [7, 8, 4]]

    final = [[1, 2, 3],
             [5, 8, 6],
             [0, 7, 4]]

    empty_tile_posi = [1, 2] # Vị trí số 0 ở hàng 1, cột 2
    solve(initial, empty_tile_posi, final)

--- Tìm thấy lời giải ---
1  2  3  
5  6  0  
7  8  4  
Cost: 3, Level: 0

1  2  3  
5  0  6  
7  8  4  
Cost: 2, Level: 1

1  2  3  
5  8  6  
7  0  4  
Cost: 1, Level: 2

1  2  3  
5  8  6  
0  7  4  
Cost: 0, Level: 3

